In [5]:
import tensorflow as tp

In [6]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
import pandas as pd

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


In [9]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [10]:
from datasets import load_dataset

dataset = load_dataset("blanchon/EuroSAT_RGB")

c:\Users\Malak\anaconda3\envs\tf_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'filename'],
        num_rows: 16200
    })
    test: Dataset({
        features: ['image', 'label', 'filename'],
        num_rows: 5400
    })
    validation: Dataset({
        features: ['image', 'label', 'filename'],
        num_rows: 5400
    })
})


In [12]:
def preprocess(example):
    image = np.array(example["image"]) / 255.0
    label = example["label"]
    return {"image": image, "label": label}

dataset = dataset.map(preprocess)

In [13]:
X_train = np.stack(dataset["train"]["image"])
y_train = np.array(dataset["train"]["label"])

X_val = np.stack(dataset["validation"]["image"])
y_val = np.array(dataset["validation"]["label"])

X_test = np.stack(dataset["test"]["image"])
y_test = np.array(dataset["test"]["label"])

In [14]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

c:\Users\Malak\anaconda3\envs\tf_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 30s 56ms/step - accuracy: 0.5280 - loss: 1.2591 - val_accuracy: 0.6054 - val_loss: 1.0339
Epoch 2/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - accuracy: 0.6991 - loss: 0.8219 - val_accuracy: 0.7157 - val_loss: 0.7888
Epoch 3/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 28s 56ms/step - accuracy: 0.7638 - loss: 0.6578 - val_accuracy: 0.7704 - val_loss: 0.6409
Epoch 4/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 30s 59ms/step - accuracy: 0.7959 - loss: 0.5659 - val_accuracy: 0.7533 - val_loss: 0.6514
Epoch 5/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 29s 57ms/step - accuracy: 0.8317 - loss: 0.4737 - val_accuracy: 0.8144 - val_loss: 0.5418
Epoch 6/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 29s 57ms/step - accuracy: 0.8557 - loss: 0.4033 - val_accuracy: 0.7880 - val_loss: 0.6481
Epoch 7/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 29s 57ms/step - accuracy: 0.8777 - loss: 0.3516 - val_accuracy: 0.8087 - val_loss: 0.5561
Epoch 8/10
507/507 ━━━━━━━━━━━━━━━━━━━━ 29s 57ms/step - accuracy: 0.8923 - loss: 0.3063 - 

In [17]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Keras Accuracy:", test_acc)

169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8370 - loss: 0.5246
Keras Accuracy: 0.8370370268821716


In [18]:
model.save("keras_cnn.h5")

In [19]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [20]:
import torch

X_train_t = torch.tensor(X_train).permute(0,3,1,2).float()
y_train_t = torch.tensor(y_train)

X_val_t = torch.tensor(X_val).permute(0,3,1,2).float()
y_val_t = torch.tensor(y_val)

X_test_t = torch.tensor(X_test).permute(0,3,1,2).float()
y_test_t = torch.tensor(y_test)

In [21]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=32)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32)

In [22]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*14*14, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [23]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.7487825751304626
Epoch 2, Loss: 0.9452939033508301
Epoch 3, Loss: 0.5356923341751099
Epoch 4, Loss: 1.202125072479248
Epoch 5, Loss: 0.3580814003944397
Epoch 6, Loss: 0.5817487239837646
Epoch 7, Loss: 0.06775002181529999
Epoch 8, Loss: 0.41660434007644653
Epoch 9, Loss: 0.4612571597099304
Epoch 10, Loss: 0.12549899518489838


In [24]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print("PyTorch Accuracy:", correct / total)

PyTorch Accuracy: 0.8361111111111111


In [25]:
torch.save(model.state_dict(), "pytorch_cnn.pth")

In [26]:
print("Keras Accuracy:", test_acc)
print ("VS.")
print("PyTorch Accuracy:", correct / total)

Keras Accuracy: 0.8370370268821716
VS.
PyTorch Accuracy: 0.8361111111111111
